# Pipeline: alle Agenten in einem Ablauf

Python prueft den Zustand je Titel und plant nur, was faellig ist:

| Schritt | faellig, wenn ... |
|---|---|
| Annual Report | ein neuerer 10-K vorliegt als der letzte Report des Agenten |
| Earnings Review (mehrere Quartale) | nach dem 10-K neue Quartalszahlen (8-K Item 2.02) gemeldet wurden, die noch nicht im Thesen-Protokoll stehen |
| Investment Committee | neuer Report/Review, News-Alarm (eskalierende Story, Divergenz) oder letztes Votum aelter als 30 Tage; bekommt Thesen, Thesen-Status und News-Signale als Kontext |
| Forecaster | woechentlich |
| News-Tracker | woechentlich (Depot) |
| Portfolio Manager | am Ende, mit der Research-Sicht je Position |

Nur US-Werte haben SEC-Filings; andere Titel laufen ueber News-Tracker und Portfolio Manager. **Erst den Plan ansehen (Zelle 4, ohne LLM), dann ausfuehren (Zelle 5).** Protokoll mit Laufzeit und Kosten: `research/pipeline_runs/`.

In [ ]:
# 1) Pfade und Imports
import os, sys, json
from pathlib import Path

ROOT = Path.cwd().resolve().parents[1]      # ...\FinRobot
AIF = ROOT / "ai_in_finance"
FR = ROOT / "finrobot_autogen"
sys.path.insert(0, str(AIF))

import pandas as pd
from IPython.display import Markdown, display
from finrobot.utils import register_keys_from_json
from lib import pipeline as pl, news_tracker as nt
register_keys_from_json(str(FR / "config_api_keys"))

In [ ]:
# 2) Modelle je Rolle
WRITER_MODEL = "anthropic/claude-haiku-5.5"      # Annual Report, Earnings Review, News-Digest, Portfolio Manager
COMMITTEE_MODEL = "anthropic/claude-haiku-5.5"
FORECAST_MODEL = "anthropic/claude-haiku-5.5"
CLASSIFIER_MODEL = "anthropic/claude-haiku-5.5"  # News-Klassifizierung (viele kleine Aufrufe)

cfgs = {"writer": pl.llm_config_for(WRITER_MODEL), "committee": pl.llm_config_for(COMMITTEE_MODEL),
        "forecast": pl.llm_config_for(FORECAST_MODEL, temperature=0.3),
        "classifier": pl.llm_config_for(CLASSIFIER_MODEL, temperature=0, max_tokens=8000)}

In [ ]:
# 3) Titel und Portfolio
# MODE = "sample": Musterportfolio (portfolios/sample_portfolio.csv); Research-Schritte nur fuer RESEARCH_TICKERS
# MODE = "transactions": deine Broker-Exporte; Research-Schritte fuer die TOP_N groessten Positionen (ohne ETFs)
MODE = "sample"
if MODE == "sample":
    PORTFOLIO_FILE = None      # Bestand ergibt sich aus der Muster-Historie (sample_portfolio.csv nur ohne Historie)
    TRANSACTION_FILES = [str(AIF / "portfolios" / "sample_transactions.csv")]   # Musterdepot mit Kaeufen/Verkaeufen seit 2023
    HOLDINGS_FILES = None
    RESEARCH_TICKERS = ["GOOG", "AMZN", "MA"]    # nur 10-K-Emittenten; jeder Titel ohne Report kostet einen Annual Report!
    watchlist = nt.watchlist_from_tickers(RESEARCH_TICKERS)
else:
    PORTFOLIO_FILE = None
    TRANSACTION_FILES = [str(AIF / "portfolios" / f) for f in ("traderepublic.csv", "ScalableCapital.csv",
                                                                 "Smartbroker1.csv", "Smartbroker2.csv")]
    HOLDINGS_FILES = []        # nur Positions-Exporte ohne Kaufhistorie
    TOP_N = 5
    watchlist = nt.top_watchlist(TRANSACTION_FILES, n=TOP_N, holdings_files=HOLDINGS_FILES)
tickers = [w["ticker"] for w in watchlist]
tickers

In [ ]:
# 4) Plan (ohne LLM): was ist faellig und warum?
steps, states = pl.plan(tickers)
pl.plan_table(steps)

In [ ]:
# 5) Ausfuehren. MAX_STEPS begrenzt die Anzahl der Agentenlaeufe (Kosten!), ONLY z.B. {"forecast", "news"}.
MAX_STEPS = 3      # erster Testlauf; danach None = alle faelligen Schritte
ONLY = None
log = pl.run_pipeline(steps, cfgs, tickers, transaction_files=TRANSACTION_FILES, portfolio_file=PORTFOLIO_FILE,
                      holdings_files=HOLDINGS_FILES,
                      watchlist=watchlist, max_steps=MAX_STEPS, only=ONLY, name=f"pipeline_{MODE}",
                      title="Portfolio Review (Pipeline)")

In [ ]:
# 6) Protokoll und Research-Sicht
display(Markdown(open(sorted((AIF / "research" / "pipeline_runs").glob("*.md"))[-1], encoding="utf-8").read()))